In [1]:
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.decomposition import PCA

set_config(transform_output="pandas")

In [2]:
tips_matrix = pd.read_csv("02_tips_matrix.csv", index_col="player_id")
tips_matrix.columns.name = "tip_id"

player_lookup = pd.read_csv("02_player_lookup.csv", index_col="player_id")

In [3]:
tips_matrix_z_scores = (tips_matrix - tips_matrix.mean()) / tips_matrix.std()

In [4]:
stats = tips_matrix.std().rename("std deviation").to_frame()

value_counts = tips_matrix.apply(pd.Series.value_counts)
value_counts_norm = value_counts / value_counts.sum()

stats["entropy (bits)"] = (-value_counts_norm * np.log2(value_counts_norm)).sum()
stats["exp2(entropy)"] = np.exp2(stats["entropy (bits)"])

stats["bracket_tip"] = stats.index.str.startswith("BRACKET").astype(int)

stats.to_csv("03_stats.csv")

In [5]:
tips_matrix.describe(percentiles=[0.5]).T.sort_values("std").to_csv("03_tip_stats.csv")

In [6]:
thingo = tips_matrix_z_scores.stack().rename("z_score").to_frame()
thingo["orig_tip_val"] = tips_matrix.stack()
thingo = thingo.dropna(subset="orig_tip_val").fillna(0).reset_index()

thingo["bracket_tip"] = thingo["tip_id"].str.startswith("BRACKET")

thingo["tip"] = "away"
thingo.loc[thingo["orig_tip_val"] == 0, "tip"] = "draw"
thingo.loc[thingo["orig_tip_val"] == 1, "tip"] = "home"
thingo.loc[thingo["orig_tip_val"] == 0.42, "tip"] = "group"
thingo.loc[(thingo["orig_tip_val"] == 1) & thingo["bracket_tip"], "tip"] = "r32"
thingo.loc[thingo["orig_tip_val"] == 2, "tip"] = "r16"
thingo.loc[thingo["orig_tip_val"] == 3, "tip"] = "quarter"
thingo.loc[thingo["orig_tip_val"] == 4, "tip"] = "semi"
thingo.loc[thingo["orig_tip_val"] == 4.42, "tip"] = "third"
thingo.loc[thingo["orig_tip_val"] == 5, "tip"] = "final"
thingo.loc[thingo["orig_tip_val"] == 6, "tip"] = "winner"

thingo["name"] = thingo["player_id"].map(player_lookup["safe_name"])

tip_bravery = (
    thingo.groupby(["tip_id", "tip", "z_score"])
    .agg(
        num_tippers=("name", "count"),
        tippers=("name", sorted),
    )
    .reset_index()
    .sort_values("tip_id")
    .sort_values(
        ["z_score", "tip_id"], key=lambda col: col.abs() if col.name == "z_score" else col, ascending=[False, True]
    )
)

tip_bravery.to_csv("03_tip_bravery.csv", index=False)

In [7]:
tips_matrix_z_score_imputed = tips_matrix_z_scores.fillna(0)  # impute missing tips as the mean
tips_matrix_z_score_imputed.index = tips_matrix_z_score_imputed.index.map(player_lookup["safe_name"])
tips_matrix_z_score_imputed.index.name = "name"

pca = PCA(n_components=0.9)
pca.fit(tips_matrix_z_score_imputed)

transformed = pca.transform(tips_matrix_z_score_imputed)

transformed.to_csv("03_pca_analysis.csv")